# Week 4 - Baseline Model Performance Evaluation

## Objective

Continue the Week 1-3 project chain without changing the underlying asset or pricing framework. This notebook loads the Week 2 processed JPM dataset, regression-checks the Week 3 chooser model, generates a historical chooser-price baseline, evaluates volatility regimes and sensitivity, benchmarks runtime and memory, and separates proxy error analysis from real listed-option market validation.

> Data boundary: public historical transaction prices for a customized chooser option are not available in the project data. JPM stock prices must never be treated as option prices. Proxy chooser errors and Cboe-listed vanilla option errors are reported separately.

## 1. Required Packages and Project Paths

In [1]:
from pathlib import Path
import json
import time
import tracemalloc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def locate_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path(r"G:\JPM-Chooser Option Pricing")]
    for candidate in candidates:
        if (candidate / "Week2" / "processed_data" / "market_data_processed.csv").exists() and (candidate / "Week 3" / "Week3.ipynb").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Week 2 dataset and Week 3 notebook.")


PROJECT_ROOT = locate_project_root()
WEEK_DIR = Path.cwd()
RESULTS_DIR = WEEK_DIR / "model_results"
FIGURES_DIR = WEEK_DIR / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

with open(WEEK_DIR / "benchmark_config.json", encoding="utf-8") as stream:
    CONFIG = json.load(stream)

print(f"Project root: {PROJECT_ROOT}")
print(f"Week 4 output: {WEEK_DIR}")

Project root: G:\JPM-Chooser Option Pricing
Week 4 output: C:\Users\CD钙奶\.codex\.chatgpt-projects\g-p-6a54aec7e7b0819197b5f2087ba33ff8\work\delivery\Week 4


## 2. Load Week 2 Processed Data

In [2]:
data_path = PROJECT_ROOT / CONFIG["week2_inputs"]["file"]
market = pd.read_csv(data_path, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

required_columns = ["Date", "Close", "Treasury_Rate", "Rolling_Volatility_20D", "VIX_Close", "Log_Return"]
missing_columns = sorted(set(required_columns) - set(market.columns))
if missing_columns:
    raise KeyError(f"Week 2 columns missing: {missing_columns}")

print(f"Loaded {len(market):,} Week 2 rows from {market['Date'].min().date()} to {market['Date'].max().date()}.")
market[required_columns].tail()

Loaded 1,760 Week 2 rows from 2018-01-02 to 2024-12-30.


           Date       Close  ...  VIX_Close  Log_Return
1755 2024-12-23  230.179596  ...  16.780001    0.003319
1756 2024-12-24  233.964584  ...  14.270000    0.016310
1757 2024-12-26  234.766006  ...  14.730000    0.003420
1758 2024-12-27  232.863876  ...  15.950000   -0.008135
1759 2024-12-30  231.077560  ...  17.400000   -0.007701

[5 rows x 6 columns]

## 3. Import and Regression-Check the Week 3 Chooser Model

In [3]:
from bsm_chooser import bsm_call_price, bsm_put_price, chooser_greeks_fd, simple_chooser_price
from week3_adapter import load_week3_functions

week3 = load_week3_functions(PROJECT_ROOT)
paper = {"S": 156.7, "K": 150.0, "r": 0.0015, "q": 0.0233, "sigma": 0.282, "T1": 0.5, "T2": 1.0}
week3_price = week3["simple_chooser_price"](**paper)
week4_price = float(simple_chooser_price(**paper))
regression_error = abs(week4_price - week3_price)
assert regression_error < 1e-10

regression_check = pd.DataFrame([{
    "Week3_Price": week3_price,
    "Week4_Vectorized_Price": week4_price,
    "Absolute_Difference": regression_error,
    "Pass": regression_error < 1e-10,
}])
regression_check

   Week3_Price  Week4_Vectorized_Price  Absolute_Difference  Pass
0    29.129924               29.129924                  0.0  True

## 4. Generate the 2018-2024 Baseline Price Series

In [4]:
K = CONFIG["contract"]["strike"]
T1 = CONFIG["contract"]["choice_time_years"]
T2 = CONFIG["contract"]["maturity_years"]
q = CONFIG["dividend_yield"]["value"]

baseline = market.dropna(subset=["Close", "Treasury_Rate", "Rolling_Volatility_20D", "VIX_Close"]).copy()
baseline["Risk_Free_Rate"] = baseline["Treasury_Rate"] / 100.0
baseline["Historical_Volatility"] = baseline["Rolling_Volatility_20D"].clip(lower=1e-6)
baseline["BSM_Call_Price"] = bsm_call_price(baseline["Close"], K, baseline["Risk_Free_Rate"], q, baseline["Historical_Volatility"], T2)
baseline["BSM_Put_Price"] = bsm_put_price(baseline["Close"], K, baseline["Risk_Free_Rate"], q, baseline["Historical_Volatility"], T2)
baseline["Chooser_BSM_Price"] = simple_chooser_price(baseline["Close"], K, baseline["Risk_Free_Rate"], q, baseline["Historical_Volatility"], T1, T2)

# Forward 20-day realized volatility is a transparent ex-post proxy, not a transaction price.
forward_vol = market["Log_Return"].rolling(20).std(ddof=1).shift(-20) * np.sqrt(252)
baseline["Forward_Realized_Volatility_20D"] = forward_vol.reindex(baseline.index)
proxy_mask = baseline["Forward_Realized_Volatility_20D"].notna()
baseline.loc[proxy_mask, "Chooser_Proxy_Reference_Price"] = simple_chooser_price(
    baseline.loc[proxy_mask, "Close"], K, baseline.loc[proxy_mask, "Risk_Free_Rate"], q,
    baseline.loc[proxy_mask, "Forward_Realized_Volatility_20D"].clip(lower=1e-6), T1, T2
)
baseline["Proxy_Pricing_Error"] = baseline["Chooser_BSM_Price"] - baseline["Chooser_Proxy_Reference_Price"]

print(f"Generated {len(baseline):,} chooser prices; {proxy_mask.sum():,} have forward-volatility proxy references.")
baseline[["Date", "Close", "Historical_Volatility", "Chooser_BSM_Price", "Chooser_Proxy_Reference_Price"]].tail()

Generated 1,740 chooser prices; 1,720 have forward-volatility proxy references.


           Date       Close  ...  Chooser_BSM_Price  Chooser_Proxy_Reference_Price
1755 2024-12-23  230.179596  ...          81.666166                            NaN
1756 2024-12-24  233.964584  ...          85.375462                            NaN
1757 2024-12-26  234.766006  ...          86.143092                            NaN
1758 2024-12-27  232.863876  ...          84.356755                            NaN
1759 2024-12-30  231.077560  ...          82.527821                            NaN

[5 rows x 5 columns]

## 5. Error Metrics: Proxy Validation Only

In [5]:
def regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = predicted - actual
    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error**2))
    denominator = np.sum((actual - actual.mean())**2)
    r2 = 1.0 - np.sum(error**2) / denominator if denominator > 0 else np.nan
    return {"N": len(actual), "MAE": mae, "RMSE": rmse, "R2": r2, "Mean_Error": error.mean()}


proxy_rows = baseline.dropna(subset=["Chooser_Proxy_Reference_Price"])
proxy_metrics = pd.DataFrame([{
    "Benchmark": "Chooser BSM with trailing 20-day volatility vs ex-post forward-volatility proxy",
    "Actual_Type": "Model-derived proxy; not an observed chooser transaction price",
    **regression_metrics(proxy_rows["Chooser_Proxy_Reference_Price"], proxy_rows["Chooser_BSM_Price"]),
}])
proxy_metrics

                                           Benchmark  ... Mean_Error
0  Chooser BSM with trailing 20-day volatility vs...  ...  -0.265177

[1 rows x 7 columns]

## 6. High-Volatility Regime Analysis

In [6]:
low_cut = CONFIG["regimes"]["low_vix_upper"]
high_cut = CONFIG["regimes"]["high_vix_lower"]
baseline["VIX_Regime"] = pd.cut(
    baseline["VIX_Close"], bins=[-np.inf, low_cut, high_cut, np.inf],
    labels=["Low (<15)", "Normal (15-25)", "High (>25)"], right=False
)

regime_summary = baseline.groupby("VIX_Regime", observed=True).agg(
    Observations=("Date", "size"),
    Mean_VIX=("VIX_Close", "mean"),
    Mean_Historical_Volatility=("Historical_Volatility", "mean"),
    Mean_Chooser_Price=("Chooser_BSM_Price", "mean"),
    Proxy_MAE=("Proxy_Pricing_Error", lambda x: x.abs().mean()),
    Proxy_RMSE=("Proxy_Pricing_Error", lambda x: np.sqrt(np.nanmean(np.square(x)))),
).reset_index()

regime_summary

       VIX_Regime  Observations  ...  Proxy_MAE  Proxy_RMSE
0       Low (<15)           490  ...   2.028167    6.635490
1  Normal (15-25)           919  ...   3.525266    6.419625
2      High (>25)           331  ...   6.994742   13.650144

[3 rows x 7 columns]

## 7. Parameter Sensitivity Analysis

In [7]:
latest = baseline.iloc[-1]
spot_grid = np.linspace(0.70 * K, 1.30 * K, 61)
vol_grid = np.linspace(0.10, 0.60, 51)
spot_mesh, vol_mesh = np.meshgrid(spot_grid, vol_grid)
surface = pd.DataFrame({
    "Spot": spot_mesh.ravel(),
    "Volatility": vol_mesh.ravel(),
    "Chooser_Price": simple_chooser_price(
        spot_mesh.ravel(), K, latest["Risk_Free_Rate"], q, vol_mesh.ravel(), T1, T2
    ),
})

rate_grid = np.linspace(max(-0.01, latest["Risk_Free_Rate"] - 0.03), latest["Risk_Free_Rate"] + 0.03, 31)
rate_sensitivity = pd.DataFrame({
    "Risk_Free_Rate": rate_grid,
    "Chooser_Price": simple_chooser_price(
        latest["Close"], K, rate_grid, q, latest["Historical_Volatility"], T1, T2
    ),
})

greeks = pd.DataFrame([chooser_greeks_fd(
    latest["Close"], K, latest["Risk_Free_Rate"], q, latest["Historical_Volatility"], T1, T2
)])
greeks

       price     delta     gamma      vega         rho
0  82.527821  0.970215  0.000445  4.346185  141.672948

## 8. Runtime and Memory Benchmark

In [8]:
def benchmark_vectorized(size, repeats=7):
    spot = np.linspace(0.70 * K, 1.30 * K, size)
    volatility = np.linspace(0.10, 0.60, size)
    timings = []
    tracemalloc.start()
    for _ in range(repeats):
        start = time.perf_counter()
        result = simple_chooser_price(spot, K, latest["Risk_Free_Rate"], q, volatility, T1, T2)
        timings.append(time.perf_counter() - start)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        "Implementation": "Vectorized Week 4",
        "N": size,
        "Median_Runtime_ms": np.median(timings) * 1000,
        "Throughput_prices_per_second": size / np.median(timings),
        "Peak_Tracked_Memory_MB": peak / 1024**2,
        "Input_Output_Array_Memory_MB": (spot.nbytes + volatility.nbytes + result.nbytes) / 1024**2,
    }


def benchmark_scalar(size, repeats=3):
    spot = np.linspace(0.70 * K, 1.30 * K, size)
    volatility = np.linspace(0.10, 0.60, size)
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        _ = [week3["simple_chooser_price"](float(s), K, float(latest["Risk_Free_Rate"]), q, float(v), T1, T2) for s, v in zip(spot, volatility)]
        timings.append(time.perf_counter() - start)
    return {
        "Implementation": "Scalar Week 3",
        "N": size,
        "Median_Runtime_ms": np.median(timings) * 1000,
        "Throughput_prices_per_second": size / np.median(timings),
        "Peak_Tracked_Memory_MB": np.nan,
        "Input_Output_Array_Memory_MB": np.nan,
    }


runtime_memory = pd.DataFrame(
    [benchmark_scalar(n) for n in [100, 1_000, 5_000]]
    + [benchmark_vectorized(n) for n in [100, 1_000, 10_000, 100_000]]
)
runtime_memory

      Implementation  ...  Input_Output_Array_Memory_MB
0      Scalar Week 3  ...                           NaN
1      Scalar Week 3  ...                           NaN
2      Scalar Week 3  ...                           NaN
3  Vectorized Week 4  ...                      0.002289
4  Vectorized Week 4  ...                      0.022888
5  Vectorized Week 4  ...                      0.228882
6  Vectorized Week 4  ...                      2.288818

[7 rows x 6 columns]

## 9. Listed JPM Option Market Sanity Check

In [9]:
snapshot = pd.read_csv(WEEK_DIR / "market_data" / "cboe_jpm_delayed_2026-09-04.csv", parse_dates=["Quote_Date", "Expiration"])
meta = CONFIG["market_snapshot"]
snapshot["Spot_Mid"] = (meta["underlying_bid"] + meta["underlying_ask"]) / 2
snapshot["Market_Mid"] = (snapshot["Bid"] + snapshot["Ask"]) / 2
snapshot["T"] = (snapshot["Expiration"] - snapshot["Quote_Date"]).dt.days / 365.0
snapshot["Risk_Free_Rate"] = meta["risk_free_rate"]
snapshot["Dividend_Yield"] = q

is_call = snapshot["Option_Type"].eq("call")
hist_sigma = meta["week2_frozen_volatility"]
snapshot["BSM_Week2_Frozen_Vol"] = np.where(
    is_call,
    bsm_call_price(snapshot["Spot_Mid"], snapshot["Strike"], snapshot["Risk_Free_Rate"], q, hist_sigma, snapshot["T"]),
    bsm_put_price(snapshot["Spot_Mid"], snapshot["Strike"], snapshot["Risk_Free_Rate"], q, hist_sigma, snapshot["T"]),
)
snapshot["BSM_Cboe_Quoted_IV"] = np.where(
    is_call,
    bsm_call_price(snapshot["Spot_Mid"], snapshot["Strike"], snapshot["Risk_Free_Rate"], q, snapshot["Quoted_IV"], snapshot["T"]),
    bsm_put_price(snapshot["Spot_Mid"], snapshot["Strike"], snapshot["Risk_Free_Rate"], q, snapshot["Quoted_IV"], snapshot["T"]),
)

market_metrics = []
for method in ["BSM_Week2_Frozen_Vol", "BSM_Cboe_Quoted_IV"]:
    for option_type, group in snapshot.groupby("Option_Type"):
        market_metrics.append({
            "Method": method,
            "Option_Type": option_type,
            "Actual_Type": "Cboe delayed listed JPM vanilla option midpoint; not chooser transaction",
            **regression_metrics(group["Market_Mid"], group[method]),
        })
    market_metrics.append({
        "Method": method,
        "Option_Type": "all",
        "Actual_Type": "Cboe delayed listed JPM vanilla option midpoint; not chooser transaction",
        **regression_metrics(snapshot["Market_Mid"], snapshot[method]),
    })
market_metrics = pd.DataFrame(market_metrics)
market_metrics

                 Method Option_Type  ...        R2  Mean_Error
0  BSM_Week2_Frozen_Vol        call  ...  0.985701   -0.242666
1  BSM_Week2_Frozen_Vol         put  ...  0.982388   -0.282624
2  BSM_Week2_Frozen_Vol         all  ...  0.984503   -0.262645
3    BSM_Cboe_Quoted_IV        call  ...  0.994982   -0.170453
4    BSM_Cboe_Quoted_IV         put  ...  0.997751    0.093503
5    BSM_Cboe_Quoted_IV         all  ...  0.996126   -0.038475

[6 rows x 8 columns]

## 10. Save Tables and Figures

In [10]:
regression_check.to_csv(RESULTS_DIR / "week3_regression_check.csv", index=False)
baseline.to_csv(RESULTS_DIR / "chooser_baseline_series_2018_2024.csv", index=False)
proxy_metrics.to_csv(RESULTS_DIR / "proxy_error_metrics.csv", index=False)
regime_summary.to_csv(RESULTS_DIR / "volatility_regime_summary.csv", index=False)
surface.to_csv(RESULTS_DIR / "spot_volatility_sensitivity.csv", index=False)
rate_sensitivity.to_csv(RESULTS_DIR / "rate_sensitivity.csv", index=False)
greeks.to_csv(RESULTS_DIR / "latest_baseline_greeks.csv", index=False)
runtime_memory.to_csv(RESULTS_DIR / "runtime_memory_benchmark.csv", index=False)
snapshot.to_csv(RESULTS_DIR / "listed_option_market_predictions.csv", index=False)
market_metrics.to_csv(RESULTS_DIR / "listed_option_market_metrics.csv", index=False)

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(11, 5.2))
ax.plot(baseline["Date"], baseline["Chooser_BSM_Price"], color="#1f4e79", linewidth=1.4, label="Chooser BSM baseline")
ax.set(title="JPM Chooser BSM Baseline Price Series (2018-2024)", xlabel="Date", ylabel="Model price ($)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "chooser_baseline_price_series.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.8))
plot_regime = regime_summary.set_index("VIX_Regime")
ax.bar(plot_regime.index.astype(str), plot_regime["Proxy_RMSE"], color=["#70ad47", "#5b9bd5", "#c00000"])
ax.set(title="Proxy Pricing RMSE by VIX Regime", xlabel="VIX regime", ylabel="RMSE ($)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "proxy_rmse_by_vix_regime.png", dpi=180)
plt.close(fig)

pivot = surface.pivot(index="Volatility", columns="Spot", values="Chooser_Price")
fig, ax = plt.subplots(figsize=(9, 5.5))
image = ax.imshow(pivot.values, origin="lower", aspect="auto", cmap="viridis", extent=[spot_grid.min(), spot_grid.max(), vol_grid.min(), vol_grid.max()])
ax.set(title="Chooser Price Sensitivity to Spot and Volatility", xlabel="JPM spot ($)", ylabel="Volatility")
fig.colorbar(image, ax=ax, label="Chooser price ($)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "spot_volatility_sensitivity.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
for method, color in [("BSM_Week2_Frozen_Vol", "#ed7d31"), ("BSM_Cboe_Quoted_IV", "#5b9bd5")]:
    ax.scatter(snapshot["Market_Mid"], snapshot[method], label=method.replace("_", " "), alpha=0.75, color=color)
limits = [0, max(snapshot["Market_Mid"].max(), snapshot[["BSM_Week2_Frozen_Vol", "BSM_Cboe_Quoted_IV"]].max().max()) * 1.05]
ax.plot(limits, limits, "k--", linewidth=1, label="Perfect agreement")
ax.set(xlim=limits, ylim=limits, title="Listed JPM Vanilla Options: Market Midpoint vs BSM", xlabel="Cboe delayed midpoint ($)", ylabel="BSM price ($)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "listed_option_market_comparison.png", dpi=180)
plt.close(fig)

summary = pd.DataFrame([
    {"Area": "Week 3 regression", "Result": f"max difference {regression_error:.3e}", "Interpretation": "Vectorized model preserves Week 3 formula"},
    {"Area": "Historical chooser series", "Result": f"{len(baseline):,} observations", "Interpretation": "Uses Week 2 spot, Treasury rate, volatility, and VIX"},
    {"Area": "Proxy error", "Result": f"MAE {proxy_metrics.loc[0, 'MAE']:.4f}; RMSE {proxy_metrics.loc[0, 'RMSE']:.4f}", "Interpretation": "Ex-post forward-volatility proxy, not transaction error"},
    {"Area": "Market sanity check", "Result": f"{len(snapshot)} listed vanilla quotes", "Interpretation": "Cboe delayed midpoints; not chooser transactions"},
    {"Area": "Complexity", "Result": "O(n) time and O(n) vector memory", "Interpretation": "Vectorized pricing supports scalable downstream ML"},
])
summary.to_csv(RESULTS_DIR / "week4_summary.csv", index=False)
print(f"Saved {len(list(RESULTS_DIR.glob('*.csv')))} result tables and {len(list(FIGURES_DIR.glob('*.png')))} figures.")
summary

Saved 11 result tables and 4 figures.


                        Area  ...                                     Interpretation
0          Week 3 regression  ...          Vectorized model preserves Week 3 formula
1  Historical chooser series  ...  Uses Week 2 spot, Treasury rate, volatility, a...
2                Proxy error  ...  Ex-post forward-volatility proxy, not transact...
3        Market sanity check  ...   Cboe delayed midpoints; not chooser transactions
4                 Complexity  ...  Vectorized pricing supports scalable downstrea...

[5 rows x 3 columns]

# Week 4 Conclusion

The Week 4 benchmark preserves the complete Week 1-3 pipeline. The historical chooser series is generated from the processed JPM spot, Treasury rate, historical volatility, and VIX features. The vectorized implementation matches the Week 3 scalar model and reduces runtime for large batches.

The reported chooser MAE and RMSE use an ex-post forward-realized-volatility proxy and must not be interpreted as transaction-price errors. The separate Cboe benchmark evaluates ordinary listed JPM calls and puts only. Public historical prices for a customized chooser option remain unavailable, so a direct chooser actual-vs-predicted benchmark cannot be claimed without proprietary OTC data.